In [1]:
import os
# Block TensorFlow import — not needed, and it causes crashes
# due to NumPy 2.x / pyarrow ABI mismatch in this Anaconda env
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_TF"] = "0"

import gc, re
from transformers import pipeline

gc.collect()
print("Cell 1 OK ✓")

Cell 1 OK ✓


In [2]:
# Load aspect classifier (zero-shot) — force CPU to avoid MPS crashes
print("Loading aspect model...")
aspect_clf = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",
    device="cpu",
)
print("Aspect model loaded ✓")

Loading aspect model...


Device set to use cpu


Aspect model loaded ✓


In [3]:
# Load Hebrew sentiment model — force CPU
print("Loading sentiment model...")
sent_clf = pipeline(
    "text-classification",
    model="dicta-il/dictabert-sentiment",
    top_k=None,
    device="cpu",
)
print("Sentiment model loaded ✓")

Loading sentiment model...


Device set to use cpu


Sentiment model loaded ✓


In [14]:
ASPECTS = ["כאב", "שינה", "אוכל"]

def split_sentences_he(text: str):
    parts = re.split(r"[.!?。\n]+", text)
    return [p.strip() for p in parts if p.strip()]

def aspect_scores(text: str):
    out = aspect_clf(text, candidate_labels=ASPECTS, multi_label=True)
    return dict(zip(out["labels"], out["scores"]))

def sentiment_probs(text: str):
    scores = sent_clf(text)[0]
    d = {x["label"].lower(): float(x["score"]) for x in scores}
    return {
        "pos": d.get("positive", 0.0),
        "neg": d.get("negative", 0.0),
        "neu": d.get("neutral", 0.0),
    }

def analyze(text: str, eps: float = 1e-9):
    sents = split_sentences_he(text)
    if not sents:
        sents = [text]

    per_sent = []
    for s in sents:
        a = aspect_scores(s)
        p = sentiment_probs(s)
        per_sent.append((a, p))

    rel = {asp: sum(a.get(asp, 0.0) for a, _ in per_sent) / len(per_sent) for asp in ASPECTS}

    overall = {
        k: sum(p[k] for _, p in per_sent) / len(per_sent)
        for k in ["pos", "neg", "neu"]
    }

    by_asp = {}
    for asp in ASPECTS:
        wsum = sum(a.get(asp, 0.0) for a, _ in per_sent)
        by_asp[asp] = {
            k: sum(a.get(asp, 0.0) * p[k] for a, p in per_sent) / (wsum + eps)
            for k in ["pos", "neg", "neu"]
        }

    string_aspects = ", \n\t".join(f"{asp}: {rel[asp]:.2f}" for asp in ASPECTS)
    
    string_by_asp = "\n\t".join(
        f"{asp}: {by_asp[asp]}" for asp in ASPECTS
    )
    message = f"Relevance:\n\t{string_aspects}\nOverall sentiment: {overall}\nSentiment by aspect:\n\t{string_by_asp}"

    return message

print("Functions defined ✓")

Functions defined ✓


In [15]:
# Test
print(analyze("ישנתי רע, אתמול בלילה התעוררתי מלא פעמים והמיטה הכאיבה לי בגב.  אכלתי ארוחת ערב מעולה."))

Relevance:
	כאב: 0.50, 
	שינה: 0.37, 
	אוכל: 0.52
Overall sentiment: {'pos': 0.4999914933547416, 'neg': 0.5000056027256505, 'neu': 2.9104119221301517e-06}
Sentiment by aspect:
	כאב: {'pos': 0.0031469131460952586, 'neg': 0.9968494287704197, 'neu': 3.6773881681472944e-06}
	שינה: {'pos': 0.0013101935441441348, 'neg': 0.9986861452425448, 'neu': 3.6802235008367826e-06}
	אוכל: {'pos': 0.9639310076161975, 'neg': 0.036066790782067495, 'neu': 2.1942310384574387e-06}


In [16]:
print(analyze("ישנתי רע, אתמול בלילה התעוררתי מלא פעמים והמיטה הכאיבה לי בגב.  אכלתי ארוחת ערב ממוצעת."))

Relevance:
	כאב: 0.80, 
	שינה: 0.37, 
	אוכל: 0.52
Overall sentiment: {'pos': 2.1764501525467495e-06, 'neg': 0.4999997623715444, 'neu': 0.4999980860289952}
Sentiment by aspect:
	כאב: {'pos': 2.0908467313476576e-06, 'neg': 0.6206778809352878, 'neu': 0.3793200513695313}
	שינה: {'pos': 1.826331375453792e-06, 'neg': 0.9935743773930203, 'neu': 0.006423815386298811}
	אוכל: {'pos': 2.5055496802043816e-06, 'neg': 0.03605670626300624, 'neu': 0.9639408162096589}
